In [ ]:




# this is used to calculate distance from roads
from scipy.ndimage import distance_transform_edt


# VEGETATION (NDVI)
# NDVI tells us how "green" or vegetated an area is plants reflect near 
# infrared light and absorb red light so we use that difference to estimate vegetation
# used chatgpt to help w this section

def compute_ndvi(images):
    ndvi_maps = []  # we will store one NDVI map per image

    for img in images:
        # get red and near-infrared channels
        red = img[:, :, 0].astype(np.float32)
        nir = img[:, :, 1].astype(np.float32)

        # NDVI formula
        ndvi = (nir - red) / (nir + red + 1e-6)

        # create empty classification map
        veg_class = np.zeros_like(ndvi)

        # convert NDVI values into categories
        veg_class[ndvi < 0.2] = 0   # low vegetation
        veg_class[(ndvi >= 0.2) & (ndvi < 0.5)] = 1  # medium
        veg_class[ndvi >= 0.5] = 2  # dense vegetation

        ndvi_maps.append(veg_class)

    return np.array(ndvi_maps)


# ROAD RISK

def compute_road_risk(masks):
    road_risk_maps = []

    for mask in masks:
        # find where roads are
        # (we assume roads = class 1... may need to check)
        road_mask = (np.argmax(mask, axis=-1) == 1).astype(np.uint8)

        # compute distance from roads
        distance = distance_transform_edt(1 - road_mask)

        # normalize (scale from 0 to 1)
        distance = distance / np.max(distance)

        # create empty risk map
        risk = np.zeros_like(distance)


        # assign risk levels based on distance
        risk[distance < 0.1] = 3   # very close = high risk
        risk[(distance >= 0.1) & (distance < 0.3)] = 2
        risk[(distance >= 0.3) & (distance < 0.6)] = 1
        risk[distance >= 0.6] = 0  # far away = low risk

        road_risk_maps.append(risk)

    return np.array(road_risk_maps)


# new labels

# vegetation labels from NDVI
vegetation_maps = compute_ndvi(preprocessed_images)

# road risk labels from masks
road_risk_maps = compute_road_risk(preprocessed_masks)

# convert CNA labels --> binary fire (fire vs no fire)
# original masks have multiple classes (background, low, high, extreme etc)
# we turn everything that is NOT background into fire
binary_masks = (np.argmax(preprocessed_masks, axis=-1) > 0).astype(np.uint8)

# expand dims so shape becomes (N, 256, 256, 1) 
binary_masks = np.expand_dims(binary_masks, axis=-1)

# check shapes
print("Vegetation maps:", vegetation_maps.shape)
print("Road risk maps:", road_risk_maps.shape)
print("Binary burn masks:", binary_masks.shape) # should be (1000, 256, 256, 1)


# split data 

(
    images_train,
    images_test_and_val,
    burn_train,
    burn_test_and_val,
    veg_train,
    veg_test_and_val,
    road_train,
    road_test_and_val,
    names_train,
    names_test_and_val,
) = train_test_split(
    preprocessed_images,
    binary_masks,
    vegetation_maps,
    road_risk_maps,
    image_names,
    test_size=0.2,
    random_state=42,
)

# split again into validation + test
(
    images_validation,
    images_test,
    burn_validation,
    burn_test,
    veg_validation,
    veg_test,
    road_validation,
    road_test,
    names_validation,
    names_test,
) = train_test_split(
    images_test_and_val,
    burn_test_and_val,
    veg_test_and_val,
    road_test_and_val,
    names_test_and_val,
    test_size=0.5,
    random_state=42,
)




#  model 

from tensorflow.keras import layers, Model

def multi_output_unet(input_shape=(256,256,3)):

    inputs = layers.Input(shape=input_shape)

    # encoder (learn features)
    c1 = layers.Conv2D(64, 3, activation='relu', padding='same')(inputs)
    c1 = layers.Conv2D(64, 3, activation='relu', padding='same')(c1)
    p1 = layers.MaxPooling2D()(c1)

    c2 = layers.Conv2D(128, 3, activation='relu', padding='same')(p1)
    c2 = layers.Conv2D(128, 3, activation='relu', padding='same')(c2)
    p2 = layers.MaxPooling2D()(c2)

    # bottleneck 
    b = layers.Conv2D(256, 3, activation='relu', padding='same')(p2)
    b = layers.Conv2D(256, 3, activation='relu', padding='same')(b)

    # decoder (rebuild image)
    u1 = layers.UpSampling2D()(b)
    u1 = layers.concatenate([u1, c2])
    c3 = layers.Conv2D(128, 3, activation='relu', padding='same')(u1)

    u2 = layers.UpSampling2D()(c3)
    u2 = layers.concatenate([u2, c1])
    c4 = layers.Conv2D(64, 3, activation='relu', padding='same')(u2)

    # outputs 
    burn_output = layers.Conv2D(1, 1, activation="sigmoid", name="burn_output")(c4)
    veg_output = layers.Conv2D(3, 1, activation="softmax", name="veg_output")(c4)
    road_output = layers.Conv2D(4, 1, activation="softmax", name="road_output")(c4)

    return Model(inputs=inputs, outputs=[burn_output, veg_output, road_output])


# build model
model = multi_output_unet()

# tell model how to learn
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss={
        "burn_output": "binary_crossentropy",
        "veg_output": "sparse_categorical_crossentropy",
        "road_output": "sparse_categorical_crossentropy",
    },
    metrics={
        "burn_output": ["accuracy"],
        "veg_output": ["accuracy"],
        "road_output": ["accuracy"],
    }
)

# print model structure
model.summary()


# training

model_fit = model.fit(
    images_train,
    {
        "burn_output": burn_train,
        "veg_output": veg_train,
        "road_output": road_train,
    },
    batch_size=8,
    epochs=25,
    validation_data=(
        images_validation,
        {
            "burn_output": burn_validation,
            "veg_output": veg_validation,
            "road_output": road_validation,
        }
    ),
    verbose=1
)


#predictions

preds = model.predict(images_test)

burn_preds = preds[0]
veg_preds = preds[1]
road_preds = preds[2]


# visualizations

def show_results(idx):
    plt.figure(figsize=(15,5))

    plt.subplot(1,4,1)
    plt.title("Image")
    plt.imshow(images_test[idx][:,:,:3])

    plt.subplot(1,4,2)
    plt.title("Burn Prediction")
    plt.imshow((burn_preds[idx] > 0.5).astype(int))

    plt.subplot(1,4,3)
    plt.title("Vegetation")
    plt.imshow(np.argmax(veg_preds[idx], axis=-1))

    plt.subplot(1,4,4)
    plt.title("Road Risk")
    plt.imshow(np.argmax(road_preds[idx], axis=-1))

    plt.show()

show_results(0)